In [ ]:
import torch
from torch import nn

import onnx
import huggingface_hub
from transformers import AutoTokenizer

import os
import json

In [ ]:
modelDir = 'models/open_clip-ViT-B-32'

# LOADS

In [ ]:
txtModelPath = huggingface_hub.hf_hub_download(
    repo_id='Marqo/onnx-open_clip-ViT-B-32',
    filename='onnx32-open_clip-ViT-B-32-laion2b_e16-textual.onnx',
    local_dir=modelDir
)
visModelPath = huggingface_hub.hf_hub_download(
    repo_id='Marqo/onnx-open_clip-ViT-B-32',
    filename='onnx32-open_clip-ViT-B-32-laion2b_e16-visual.onnx',
    local_dir=modelDir
)
tokenizerPath = huggingface_hub.hf_hub_download(
    repo_id='openai/clip-vit-base-patch32',
    filename='tokenizer.json',
    local_dir=modelDir
)
tokenizerConfigPath = huggingface_hub.hf_hub_download(
    repo_id='openai/clip-vit-base-patch32',
    filename='tokenizer_config.json',
    local_dir=modelDir
)
configPath = huggingface_hub.hf_hub_download(
    repo_id='openai/clip-vit-base-patch32',
    filename='config.json',
    local_dir=modelDir
)

In [ ]:
with open(f'{modelDir}/config.json') as f:
    config = json.load(f)
imgSz = config['vision_config']['image_size']
txtSz = config['text_config']['max_position_embeddings']
embDim = config['projection_dim']

print('image_size:', imgSz)
print('max_position_embeddings:', txtSz)
print('projection_dim:', embDim)

# DEFINE PREPROCESS MODEL

In [ ]:
class Preprocess(nn.Module):
    def __init__(self, size, normalize):
        super().__init__()
        self.norm = normalize
        self.sz = size
        
    def forward(self, x):
        x = torch.reshape(x, (self.sz*self.sz, 3)).T
        x = torch.reshape(x, (3, self.sz, self.sz))
        x = self.norm(x)
        x = x.unsqueeze(0)

        return x

In [ ]:
mean = torch.reshape(torch.tensor([
                [0.48145466] * imgSz*imgSz, 
                [0.45782750] * imgSz*imgSz, 
                [0.40821073] * imgSz*imgSz]), 
    (3, imgSz, imgSz))
std = torch.reshape(torch.tensor([
                [0.26862954] * imgSz*imgSz, 
                [0.26130258] * imgSz*imgSz, 
                [0.27577711] * imgSz*imgSz]), 
    (3, imgSz, imgSz))

normalize = lambda x: (x - mean) / std

In [ ]:
preprocess = Preprocess(imgSz, normalize)
preprocess.eval()

# COMBINE VISUAL MODELS

In [ ]:
visualOnnx = onnx.load(visModelPath)

In [ ]:
onnxprog = torch.onnx.export(preprocess, 
                             torch.ones(1, 3*imgSz*imgSz, dtype=torch.float32))

In [ ]:
try:
    os.remove(f'{modelDir}/preprocess.onnx')
except:
    pass
onnxprog.save(f'{modelDir}/preprocess.onnx')

In [ ]:
preprocessOnnx = onnx.load(f'{modelDir}/preprocess.onnx')

In [ ]:
# ensure opset and ir version matches
intermediate = onnx.version_converter.convert_version(visualOnnx, preprocessOnnx.opset_import[0].version)
intermediate.ir_version = preprocessOnnx.ir_version

In [ ]:
combinedOnnx = onnx.compose.merge_models(preprocessOnnx, intermediate,
    prefix1='p',
    prefix2='v',
    io_map=[(
    preprocessOnnx.graph.output[0].name,
    visualOnnx.graph.input[0].name)])

# SAVE MODELS

In [ ]:
try:
    os.remove(f'{modelDir}/visual.onnx')
except:
    pass
onnx.save(combinedOnnx, f'{modelDir}/visual.onnx')

In [ ]:
!cp {modelDir}"/onnx32-open_clip-ViT-B-32-laion2b_e16-textual.onnx" {modelDir}/textual.onnx

# SETUP JSON CONFIG

In [ ]:
json = f"""
{{
  "mlPath": "",
  "txtInputDType": "int32",
  "visInputDType": "float32",
  "imgSize": {imgSz},
  "contextLen": {txtSz},
  "embeddingLen": {embDim},
  "similarity": "cosineSimilarity"
}}
"""

In [ ]:
with open(f'{modelDir}/mlconfig.json', 'w') as f:
    f.write(json)